# Stage 3 - DPO Preference Alignment

We align the SFT model with 50+ `(prompt, chosen, rejected)` preference triples
using Direct Preference Optimization, improving tone, helpfulness, and safety.

**Input:** `data/preference_dataset.jsonl` + `models/sft_adapter/`
**Output:** LoRA adapter -> `models/dpo_adapter/`

In [ ]:
%%capture
# Clean, version-consistent install. Let Unsloth pin its own compatible
# transformers/trl/peft versions - do NOT force-upgrade trl separately.
!pip install --upgrade unsloth unsloth_zoo
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Get the project code + datasets into Colab.
# Replace <your-username> with your GitHub username.
!git clone https://github.com/<your-username>/customer-support-assistant.git
%cd customer-support-assistant

import os
DATA = "data"
MODELS = "models"
os.makedirs(MODELS, exist_ok=True)
print("cwd:", os.getcwd())

## 1. Load the SFT model (from the saved adapter folder)

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()   # must run before building the DPOTrainer

import torch, os
MAX_SEQ_LENGTH = 2048

sft_dir = "models/sft_adapter"
start = sft_dir if os.path.isdir(sft_dir) else "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
print("loading from:", start)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = start,          # loading the adapter folder rebuilds base + SFT LoRA
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

# Ensure the LoRA weights are trainable for DPO, and gradients can flow.
for _n, _p in model.named_parameters():
    if "lora_" in _n:
        _p.requires_grad_(True)
model.enable_input_require_grads()

## 2. Apply chat template and format the preference dataset

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

def fmt(ex):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": ex["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    return {
        "prompt":   prompt,
        "chosen":   ex["chosen"]   + tokenizer.eos_token,
        "rejected": ex["rejected"] + tokenizer.eos_token,
    }

pref = load_dataset("json", data_files="data/preference_dataset.jsonl", split="train").map(fmt)
print(">>> preference examples:", len(pref))
print(pref[0]["prompt"][:300])

## 3. Configure and run DPO

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,               # frozen base used as implicit reference
    processing_class = tokenizer,
    train_dataset = pref,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 2,
        learning_rate = 5e-6,
        beta = 0.1,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage3",
        report_to = "none",
    ),
)
dpo_trainer.train()

## 4. Test the model AFTER DPO

In [ ]:
FastLanguageModel.for_inference(model)

def answer(q, n=200):
    msgs = [{"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=n,
                         temperature=0.7, top_p=0.9, do_sample=True,
                         eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

for q in ["How can I cancel my order?",
          "My order says delivered but I did not receive it.",
          "Someone may have accessed my account."]:
    print("Q:", q); print("A:", answer(q)); print("-"*70)

## 5. Save the DPO-aligned adapter

In [ ]:
model.save_pretrained("models/dpo_adapter")
tokenizer.save_pretrained("models/dpo_adapter")
print("Saved -> models/dpo_adapter")

# (Optional) merged 16-bit export for local/CPU inference:
# model.save_pretrained_merged("models/dpo_merged", tokenizer, save_method="merged_16bit")

### Stage 3 done
Fill in `reports/final_evaluation.md` (Base vs SFT vs DPO), then use
`src/inference.py` to chat with the final model.